In [ ]:
from datetime import datetime

# Widgets (idempotent pour relances manuelles)
for w in ["run_id", "export_base"]:
    try:
        dbutils.widgets.remove(w)
    except Exception:
        pass

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("export_base", "/Volumes/workspace/default/exports")

run_id = dbutils.widgets.get("run_id") or datetime.now().strftime("%Y%m%d_%H%M%S")
export_base = dbutils.widgets.get("export_base")

export_dir = f"{export_base}/runs/{run_id}/albums_manga_clean_export"
final_csv  = f"{export_base}/runs/{run_id}/albums_manga_clean.csv"


In [0]:
from pyspark.sql import functions as F

SOURCE_TABLE = "default.albums_raw"
TARGET_TABLE = "default.albums_manga_clean"

In [0]:
# 1) Charger la table brute
df = spark.table(SOURCE_TABLE)

In [0]:
# 2) Normaliser categories (trim + lowercase) pour filtrage + extraction sous-catégorie
df = df.withColumn("cat_norm", F.lower(F.trim(F.col("categories"))))

In [0]:
# 3) Filtre robuste : manga/mangas au début + option "inspiration manga"
is_manga = (
    F.col("cat_norm").rlike(r"^mangas?\b") |
    (F.col("cat_norm") == F.lit("inspiration manga"))
)
df = df.filter(is_manga)


In [0]:
# 4) Créer la sous-catégorie (manga_subcat)
# - "manga"/"mangas" -> general
# - "manga - xxx"/"mangas - xxx" -> xxx
# - "inspiration manga" -> inspiration
df = df.withColumn(
    "manga_subcat",
    F.when(F.col("cat_norm").rlike(r"^mangas?\s*-\s*"),
           F.regexp_extract(F.col("cat_norm"), r"^mangas?\s*-\s*(.*)$", 1))
     .when(F.col("cat_norm").isin("manga", "mangas"), F.lit("general"))
     .when(F.col("cat_norm") == "inspiration manga", F.lit("inspiration"))
     .otherwise(F.lit(None))
)

In [0]:
# 5) Sélection des colonnes demandées (+ colonnes raw pour typage)
df = df.select(
    "titre",
    "note",
    "nb_notes",
    "auteur",
    "publisher",
    "synopsis",
    "categories",
    "manga_subcat"
)

In [0]:
# 6) Nettoyage texte (trim, espaces multiples, vides -> null)
def clean_text_col(c):
    # - trim
    # - espaces multiples -> 1 espace
    # - chaîne vide -> null
    cleaned = F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")
    return F.when(F.col(c).isNull(), F.lit(None)) \
            .when(cleaned == F.lit(""), F.lit(None)) \
            .otherwise(cleaned) \
            .alias(c)

df = df.select(
    clean_text_col("titre"),
    F.col("note").alias("note_raw"),
    F.col("nb_notes").alias("nb_notes_raw"),
    clean_text_col("auteur"),
    clean_text_col("publisher"),
    clean_text_col("synopsis"),
    clean_text_col("categories"),
    clean_text_col("manga_subcat")
)


In [0]:
# 7) Nettoyage + typage NOTE (barème confirmé 0–5)
# - virgule décimale -> point
# - suppression caractères parasites
# - cast double
note_num = F.regexp_replace(F.trim(F.col("note_raw")), ",", ".")
note_num = F.regexp_replace(note_num, r"[^0-9\.\-]", "")
df = df.withColumn("note", note_num.cast("double"))

# borne 0..5 (confirmée par tes stats)
df = df.withColumn(
    "note",
    F.when((F.col("note").isNotNull()) & (F.col("note") >= 0) & (F.col("note") <= 5), F.col("note"))
     .otherwise(F.lit(None).cast("double"))
)

In [0]:
#  note sur 10 
df = df.withColumn(
    "note_10",
    F.when(F.col("note").isNotNull(), (F.col("note") * 2).cast("double"))
     .otherwise(F.lit(None).cast("double"))
)

In [0]:
# 8) Nettoyage + typage NB_NOTES (int)
# - enlève espaces
# - extrait digits
nb = F.regexp_replace(F.trim(F.col("nb_notes_raw")), r"\s+", "")
nb = F.regexp_extract(nb, r"(\d+)", 1)
df = df.withColumn("nb_notes", nb.cast("int"))

df = df.withColumn(
    "nb_notes",
    F.when((F.col("nb_notes").isNotNull()) & (F.col("nb_notes") >= 0), F.col("nb_notes"))
     .otherwise(F.lit(None).cast("int"))
)

In [0]:
# 9) Règles qualité
# - titre obligatoire
df = df.filter(F.col("titre").isNotNull() & (F.length(F.col("titre")) > 0))

In [0]:
# 10) Déduplication

df = df.dropDuplicates(["titre", "auteur", "publisher"])

In [0]:
# 11) Colonnes finales 
df_out = df.select(
    "titre",
    "note",
    "note_10",
    "nb_notes",
    "auteur",
    "publisher",
    "synopsis",
    "categories",
    "manga_subcat"
)


In [0]:
# 12) Écrire la table Delta clean
(df_out.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true") 
     .saveAsTable(TARGET_TABLE))

print(f"✅ Table créée/écrasée : {TARGET_TABLE}")


✅ Table créée/écrasée : default.albums_manga_clean


In [0]:
# -----------------------------
# 13) Contrôles qualité 
# -----------------------------
clean = spark.table(TARGET_TABLE)

print("Rows CLEAN:", clean.count())
clean.printSchema()

# KPI: bornes note
clean.select(
    F.min("note").alias("min_note"),
    F.max("note").alias("max_note"),
    F.avg("note").alias("avg_note")
).show()

# KPI: null rates
nulls = clean.select([
    F.mean(F.col(c).isNull().cast("int")).alias(f"null_rate__{c}")
    for c in ["note", "nb_notes", "auteur", "publisher", "synopsis", "manga_subcat"]
])
display(nulls)

# Top sous-catégories
(clean.groupBy("manga_subcat")
      .count()
      .orderBy(F.desc("count"))
      .show(50, truncate=False))

# Aperçu (tu peux afficher toutes les colonnes)
display(clean.limit(20))

Rows CLEAN: 27392
root
 |-- titre: string (nullable = true)
 |-- note: double (nullable = true)
 |-- note_10: double (nullable = true)
 |-- nb_notes: integer (nullable = true)
 |-- auteur: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- synopsis: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- manga_subcat: string (nullable = true)

+--------+--------+------------------+
|min_note|max_note|          avg_note|
+--------+--------+------------------+
|     1.4|     5.0|3.9291444539982807|
+--------+--------+------------------+



null_rate__note,null_rate__nb_notes,null_rate__auteur,null_rate__publisher,null_rate__synopsis,null_rate__manga_subcat
0.9575423481308412,0.8509418808411215,0.0,0.0,0.0,0.0


+------------+-----+
|manga_subcat|count|
+------------+-----+
|shôjo       |15208|
|seinen      |9515 |
|yaoi        |1253 |
|josei       |518  |
|hentai      |377  |
|ecchi       |107  |
|inspiration |97   |
|kodomo      |84   |
|general     |80   |
|anime comics|76   |
|horreur     |64   |
|yonkoma     |12   |
|dojinshi    |1    |
+------------+-----+



titre,note,note_10,nb_notes,auteur,publisher,synopsis,categories,manga_subcat
"Gurren lagann, Tome 2",null,null,null,"Mori, Kotaro",Glénat Manga,"Kamina, Yoko et Simon ont atteint la surface. Ils pilotent maintenant Gurren et Lagann, les ganmen qu’ils ont dérobés aux homme-animaux. Après avoir affronté l’un de ces puissants guerriers, Viral, ils font la connaissance des habitants du village d’Adaï et de leur mystérieux révérend. Mais alors qu’ils reprennent la route, une menace encore plus grave se profile à l’horizon. source: éditeur",mangas - shôjo,shôjo
"Friends games, Tome 18",null,null,null,"Yamaguchi, Mikoto",Soleil Productions,nan,mangas - seinen,seinen
"Don't call it mystery, Tome 6",null,null,null,"Tamura, Yumi",Noeve Grafx,nan,mangas - shôjo,shôjo
"Les vacances de Jésus et Bouddha, Tome 6",null,null,null,"Nakamura, Hikaru",Kurokawa,"Quel est le disciple préféré de Jésus ? Quel est le sport le plus pratiqué au ciel ? Pourquoi Longinus est entré dans l'Histoire ? Comment le manga célestement populaire ""Eveille-toi ! Ananda"" est crée ? Ou encore quelles chansons les deux sauveurs de l'humanité choisissent lorsqu'ils vont au karaoké ! Avec en bonus, un reportage photo exclusif à la plage !! Jésus pourra-t-il enfin nager ? Le régime de Bouddha a-t-il porté ses fruits !? source: éditeur",mangas - seinen,seinen
"Je veux t'aimer jusqu'à ta mort, Tome 2",null,null,null,"Aono, Nachi",Meian,"Shîna et Mimi s'habituent doucement à leur cohabitation et deviennent amies. Le duo se rapproche peu à peu et devient inséparable; que ce soit en classe, pendant leurs congés ou leurs événements scolaires. De son côté, Mimi enchaîne les allers-retours entre l'école et le champ de bataille, tandis que la mort semble toujours rôder au plus près d'elle. Cependant, le secret qu'elle avoue à Shîna déconcerte plus que jamais sa nouvelle amie...de bataille. source: éditeur",mangas - yaoi,yaoi
"L'enfant en moi, Tome 4",null,null,null,"Mamoru, Aoi",Kana,nan,mangas - shôjo,shôjo
Attacker you ! - Jeanne et Serge ( アタッカーＹＯＵ！) n°2,null,null,null,"Koizumi, Shizuo",Kodansha,nan,mangas - shôjo,shôjo
Catch X mama (リン×ママ) n°2,null,null,null,"Manabe, Johji",Takeshobo,nan,mangas - hentai,hentai
"Bird cage castle, Tome 2",null,null,null,"Minami, Toutarou",Bamboo Éditions,"Toujours prisonniers du “Château de la cage aux oiseaux”, six lycéens sont désormais divisés en deux groupes : tandis qu’un couple se retrouve au sommet de l’édifice, les deux autres explorent ses fondations et se retrouvent nez à nez avec un binôme sorti de nulle part. Sauront-ils unir leur force pour trouver le chemin de la sortie ? Ou l’apparition de ces nouveaux venus sèmera-t-elle la confusion au sein du groupe ? source: éditeur",mangas - seinen,seinen
Hito ha ika ni shite shinda noka ?,null,null,null,"Shibatani, Ken",Futabasha,nan,mangas - seinen,seinen


In [0]:
df = spark.table(TARGET_TABLE)

(df.coalesce(1)
   .write.mode("overwrite")
   .option("header", True)
   .csv(export_dir)
)

dbutils.fs.ls(export_dir)



[FileInfo(path='dbfs:/Volumes/workspace/default/exports/albums_manga_clean_export/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1765827887000),
 FileInfo(path='dbfs:/Volumes/workspace/default/exports/albums_manga_clean_export/_committed_2292586811456418807', name='_committed_2292586811456418807', size=113, modificationTime=1765827887000),
 FileInfo(path='dbfs:/Volumes/workspace/default/exports/albums_manga_clean_export/_started_2292586811456418807', name='_started_2292586811456418807', size=0, modificationTime=1765827885000),
 FileInfo(path='dbfs:/Volumes/workspace/default/exports/albums_manga_clean_export/part-00000-tid-2292586811456418807-f1c70c8e-ce27-49ab-9e25-81b2a69200dd-215-1-c000.csv', name='part-00000-tid-2292586811456418807-f1c70c8e-ce27-49ab-9e25-81b2a69200dd-215-1-c000.csv', size=9741609, modificationTime=1765827886000)]

In [0]:
files = dbutils.fs.ls(export_dir)
part_csv = [f.path for f in files if f.path.endswith(".csv")][0]

dbutils.fs.cp(part_csv, final_csv, True)

final_csv


'/Volumes/workspace/default/exports/albums_manga_clean.csv'

In [ ]:
dbutils.notebook.exit(final_csv)
